In [2]:
## Pothan Tang, 8/15/25
## Compute RF potential taking into account M2 grounding 

import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt

## region of interest
um = c.um
res = 1*um; # resolution of potential 
x_max = 100*um;
y_max = 100*um;
z_max = 200*um;
x = np.linspace(-1*x_max,x_max, int(2*x_max/res+1), endpoint=True); # -1cm<=x<=1cm 
y = np.linspace(-1*y_max,y_max, int(2*y_max/res+1), endpoint=True);  # -1cm<=y<=1cm
z = np.linspace(0,z_max, int(z_max/res+1), endpoint=True);  # 0cm<=z<=1cm
phi_rf = np.zeros((int(2*x_max/res+1),int(2*y_max/res+1),int(z_max/res+1))); # potential initialized to zero

## Functions
# (x,y,z) = coordinate of sample
# (xi,yi,z0) = ith corner coordinate of electrode
def potential_term(x,y,z,xi,yi,z0):
    num = (xi-x)*(yi-y); # numerator
    den = (z-z0)*math.sqrt((z-z0)**2+(xi-x)**2+(yi-y)**2); # denominator
    return num/den

def dc_potential_single_electrode(x,y,z,x1,y1,x2,y2,z0,v):
    term1 = math.atan(potential_term(x,y,z,x2,y2,z0))
    term2 = math.atan(potential_term(x,y,z,x1,y2,z0))
    term3 = math.atan(potential_term(x,y,z,x2,y1,z0))
    term4 = math.atan(potential_term(x,y,z,x1,y1,z0))
    return (v/(2*math.pi))*(term1-term2-term3+term4)

## Compute Potential
for p in range(0,len(x)):
    xc = x[p]
    for q in range (0,len(y)):
        yc = y[q]
        for r in range (0,len(z)):
            zc = z[r]
            phi_1 = dc_potential_single_electrode(xc,yc,zc,c.x11,c.y11,c.x21,c.y21,0,c.vrf); #RF electrodes located at z=0
            phi_2 = dc_potential_single_electrode(xc,yc,zc,c.x12,c.y12,c.x22,c.y22,0,c.vrf)
            phi_rf[p][q][r]+=(phi_1+phi_2)

phi_rf_reshaped = phi_rf.reshape(int(2*x_max/res+1),-1)
np.savetxt("rf_potential_max_zoomed", phi_rf_reshaped)

## Compute Pseudopotential (J)
"""
# Method 1: numerical RF gradient
Ex,Ey,Ez = np.gradient(-1*phi_rf,res,res,res); # Electric field components
E_squared = Ex**2+Ey**2+Ez**2; # Electric field strength squared
pseudo2 = (c.Z**2*c.e**2)/(4*c.m*c.omega**2)*E_squared
# pseudo_reshaped = pseudo.reshape(int(2*x_max/res+1),-1)
"""
pseudo = np.zeros((int(2*x_max/res+1),int(2*y_max/res+1),int(z_max/res+1)))

# Method 2: analytical RF gradient
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def pseudopotential(x,y,z,m=c.m,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z)
    return ((c.Z**2*c.e**2)/(4*c.m*c.omega**2)) * (VRF/math.pi)**2 * (divypart**2+divzpart**2)

for p in range(0,len(x)):
    xc = x[p]
    for q in range (0,len(y)):
        yc = y[q]
        for r in range (0,len(z)):
            zc = z[r]
            pseudo[p][q][r] += pseudopotential(xc,yc,zc)

pseudo_reshaped = pseudo.reshape(int(2*x_max/res+1),-1)
np.savetxt("pseudo_potential_zoomed", pseudo_reshaped)

"""
print(pseudo[100,100,:]); # check that pseudopotential is similar for the two methods of calculation
print(pseudo2[100,100,:])
print(pseudopotential(0,0,70.00357*um)); # check that pseudopotential is 0 at ion location
"""

/tmp/ipykernel_15038/1158729908.py:26: RuntimeWarning: divide by zero encountered in scalar divide
  return num/den


'\nprint(pseudo[100,100,:]); # check that pseudopotential is similar for the two methods of calculation\nprint(pseudo2[100,100,:])\nprint(pseudopotential(0,0,70.00357*um)); # check that pseudopotential is 0 at ion location\n'